In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
import sklearn
from sklearn import datasets
sklearn.set_config(display='text')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, Normalizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score

# 앙상블(Ensemble) 알고리즘

앙상블 알고리즘은 여러 개의 분류 모델을 조합해서 더 좋은 성능을 얻어내는 방법이다.  
k최근접 이웃, 서포트 벡터 머신, 의사결정 트리, 나이브 베이즈 모델을 실습했는데 지금까지 실습한 여러 분류 모델을 조합해서 단일 모델보다 더 좋은 성능을 낸다.

앙상블 알고리즘은 크게 독립적 앙상블과 의존적 앙상블로 나눌 수 있다.

`독립적 앙상블`  
독립적 앙상블 방법에서 사용하는 개별 분류기들을 서로 독립적인 알고리즘이므로 각 분류기는 `서로 다른 머신러닝 알고리즘을 사용`할 수 있다.  
각 분류기가 독립적이므로 효과적으로 `병렬화` 할 수 있는 장점이 있다.

`의존적 앙상블`  
의존적 앙상블 방법은 독립적 앙상블 방법과는 달리 개별 분류기들이 서로 독립이 아닌 경우를 의미하며 의존적 앙상블 방법 중 가장 유명한 방법은 부스팅이다.

## 배깅(bagging)과 랜덤 포레스트(random forest)

배깅은 부트스트랩(Bootstrap)과 어그리게이팅(집계, AGGregatING)에서 왔고 개별 분류기들의 결과를 종합하여 최종 분류기의 성능을 향상시키는 방법이다.  
상당히 연주가 힘든 바이올린 연주곡을 두 세 명의 초급 바이올린 연주자가 나누어 연주함으로써 한 명의 중급 바이올린 연주자가 연주하는 것 보다 더 낳은 연주를 할 수 있는 것과 유사하다. 중급 바이올린 연주자는 나름 잘 학습됐지만 학습 데이터에 과적합된 의사결정 트리라 볼 수 있다. 의사결정 트리는 최대 단점은 쉽게 과적합이 된다는것인데 배깅은 과적합이 되기 쉬은 모델에 상당히 적합한 앙상블 기법이다.

배깅은 한 가지 분류 모델을 여러 개를 만들어서 서로 다른 학습 데이터로 학습시킨 후(부트스트랩) 동일한 테스트에 대한 서로 다른 예측값들을 투표를 통해(어그리게이팅) 가장 높은 예측값으로 최종 결론을 내리는 기법으로 개별 분류기들의 결과를 종합하여 최종 분류기의 성능을 향상시키는 방법이다.

개별 분류기들이 동일한 학습 데이터로 학습하는 보팅(voting)과는 달리 배깅은 오리널 학습 데이터셋에서 샘플을 뽑아서 학습한다. 부트스트랩이란 중복을 허용한 랜덤 샘플링 방식을 의미하고 개별 분류 모델의 결과값을 모아서 다수결 투표를 통해서 최종 예측을 하게된다.

<img src="./bootstrap.png" width="1500" align="left" />
<img src="./aggregating1.png" width="1500" align="left" />
<img src="./aggregating2.png" width="1500" align="left" />
<img src="./aggregating3.png" width="1500" align="left" />
<img src="./boosting1.png" width="1500" align="left" />
<img src="./boosting2.png" width="1500" align="left" />

랜덤 포레스트는 여러 개의 개별 분류기인 의사결정 트리를 토대로 예측한 결과를 종합해서 전체 예측 정확도를 높이는 방법으로 여러 의사결정 트리를 배깅을 적용해서 예측을 실행한다.

배깅은 모든 분류 모델에 적용이 가능하지만 특히 과적합되기 쉬운 의사결정 트리에 사용하면 확실히 과적합을 줄여 성능이 높아지는 혜택을 보기때문에 많은 의사결정 트리 모델의 개선을 이루었고 여러 개의 나무들이 모여있다는 개념에서 랜덤 포레스트라는 이름이 생겨났다.

의사결정 트리에서 최적의 특징으로 트리를 분기하는 반면 랜덤 포레스트는 각 노드에 주어진 데이터를 샘플링해서 일부 데이터를 제외하거나 또는 중복시켜서 최적의 특징을 찾으며 트리를 분기한다. 이러한 과정에서 랜덤 포레스트는 편향을 증가시켜 과적합의 위험을 감소시킨다.

## 와인 데이터를 앙상블(랜덤 포레스트) 알고리즘에 사용할 데이터를 준비한다.

In [2]:
raw_data = datasets.load_wine()

xData = raw_data.data
yData = raw_data.target

x_train, x_test, y_train, y_test = train_test_split(xData, yData, random_state=0)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

## 랜덤 포레스트 모델을 만들고 학습 시킨다.

랜덤 포레스트 알고리즘을 사용하기 위해 RandomForestClassifier를 import 한다.

In [3]:
from sklearn.ensemble import RandomForestClassifier

In [4]:
model = RandomForestClassifier(random_state=0, max_depth=3)
model.fit(x_train, y_train)

RandomForestClassifier(max_depth=3, random_state=0)

## 학습된 랜덤 포레스트 모델을 테스트 데이터로 예측한다.

In [5]:
predict = model.predict(x_test)
print(predict)

[0 2 1 0 1 1 0 2 1 1 2 2 0 1 2 1 0 0 2 0 1 0 0 1 1 1 1 1 1 2 0 0 1 0 0 0 2
 1 1 2 0 0 1 1 1]


In [6]:
predict_proba = model.predict_proba(x_test)
print(predict_proba)

[[0.96648677 0.02191666 0.01159657]
 [0.00803061 0.13210104 0.85986835]
 [0.25908403 0.71894594 0.02197003]
 [0.91275927 0.06686862 0.0203721 ]
 [0.11200941 0.7827148  0.10527579]
 [0.3969256  0.52087364 0.08220076]
 [0.91250886 0.05988635 0.02760478]
 [0.00704412 0.097031   0.89592488]
 [0.01367628 0.97522275 0.01110097]
 [0.020483   0.95695953 0.02255747]
 [0.15000557 0.15084747 0.69914696]
 [0.00347269 0.12989998 0.86662733]
 [0.97948677 0.01808333 0.0024299 ]
 [0.27908403 0.69894594 0.02197003]
 [0.02337269 0.06464707 0.91198023]
 [0.03019417 0.96288857 0.00691726]
 [0.87534568 0.08792046 0.03673386]
 [0.97948677 0.01808333 0.0024299 ]
 [0.0235357  0.39794663 0.57851767]
 [0.97537566 0.02219444 0.0024299 ]
 [0.45644253 0.48099515 0.06256232]
 [0.8299621  0.13313565 0.03690224]
 [0.56989761 0.34479756 0.08530484]
 [0.05314679 0.93069654 0.01615667]
 [0.01663471 0.69830437 0.28506092]
 [0.0290516  0.93777952 0.03316888]
 [0.23494735 0.65528833 0.10976432]
 [0.01229015 0.97640084 0.01

## 모델을 평가한다.

In [7]:
classification = classification_report(y_test, predict, target_names=raw_data.target_names)
print(classification)

              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        16
     class_1       1.00      0.95      0.98        21
     class_2       0.89      1.00      0.94         8

    accuracy                           0.98        45
   macro avg       0.96      0.98      0.97        45
weighted avg       0.98      0.98      0.98        45

